# SQLAlchemy - Como Criar Banco de Dados Usando Apenas Python <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" height="45" />🧱

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![SQLAlchemy](https://img.shields.io/badge/SQLAlchemy-D71F00?style=flat-square&logo=sqlalchemy&logoColor=white)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-sqlalchemy%20%7C%20orm%20%7C%20sqlite-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Intermediário-yellow)
![Pré-req](https://img.shields.io/badge/pré--req-pyodbc%20%7C%20classes-purple)
![Biblioteca](https://img.shields.io/badge/requer-sqlalchemy-orange)

> Até aqui, toda tabela foi criada escrevendo `CREATE TABLE` na mão. O **SQLAlchemy** inverte isso: você descreve a tabela como uma **classe Python**, e ele mesmo gera e executa o SQL por trás — inclusive criando o arquivo do banco do zero, sem precisar de nenhum servidor rodando (aqui, um banco **SQLite**, que é só um arquivo).
>
> Diferente dos notebooks anteriores, o código de modelo deste notebook mora em `../python/modelos.py` — um módulo Python separado, do jeito que um projeto real organizaria isso (models fora do notebook).

## 📋 Conteúdo

1. [Onde Fica Cada Coisa](#-1-onde-fica-cada-coisa)
2. [Os Modelos em python/modelos.py](#-2-os-modelos-em-pythonmodelospy)
3. [Criando o Banco](#-3-criando-o-banco)
4. [Create — Inserindo com Objetos Python](#-4-create-inserindo-com-objetos-python)
5. [Read — Consultando com a ORM](#-5-read-consultando-com-a-orm)
6. [Update e Delete](#-6-update-e-delete)


## 🗂️ 1. Onde Fica Cada Coisa

| Arquivo 🔑 | Papel 🔓 |
|---|---|
| `python/__init__.py` | marca a pasta `python/` como um pacote importável |
| `python/modelos.py` | classes `Categoria` e `Produto`, e as funções `criar_engine()` / `nova_sessao()` |
| `sql/schema_sqlalchemy.sql` | o `CREATE TABLE` que o SQLAlchemy gera por trás — só de referência |
| `spec/hashtag_sqlalchemy.db` | o arquivo do banco de verdade, criado ao rodar este notebook |
| `ipynb/151-...ipynb` | este notebook, que importa `modelos` e usa tudo isso |

## 🏛️ 2. Os Modelos em python/modelos.py

Cada classe em `modelos.py` herda de `Base` (a base declarativa do SQLAlchemy) e vira uma tabela — o nome da classe vira a tabela (via `__tablename__`), e cada `Column` vira uma coluna.

| Conceito SQL 🔑 | Equivalente SQLAlchemy 🔓 |
|---|---|
| `CREATE TABLE` | uma classe Python herdando de `Base` |
| Coluna | `Column(Tipo, ...)` |
| Chave primária | `Column(Integer, primary_key=True)` |
| Chave estrangeira | `Column(Integer, ForeignKey("outra_tabela.id"))` |
| `JOIN` implícito | `relationship(...)` — navega entre objetos sem escrever SQL |

O import abaixo só funciona porque `python/` está um nível acima de `ipynb/` — `sys.path.append` aponta pra lá.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("../python").resolve()))

from modelos import Base, Categoria, Produto, criar_engine, nova_sessao, CAMINHO_BANCO
from cores import *

print(f"{CinzaClaro}Modelos importados de:{Reset} {VerdeClaro}modelos.py{Reset}")
print(f"{CinzaClaro}Tabelas mapeadas:{Reset} {MagentaClaro}{list(Base.metadata.tables.keys())}{Reset}")


Modelos importados de: modelos.py
Tabelas mapeadas: ['categorias', 'produtos']


## 🏗️ 3. Criando o Banco

`criar_engine()` (definida em `modelos.py`) chama `Base.metadata.create_all(engine)` — isso lê todas as classes que herdam de `Base` e cria as tabelas correspondentes no arquivo SQLite, **sem nenhum SQL escrito aqui**.

In [2]:
if CAMINHO_BANCO.exists():
    CAMINHO_BANCO.unlink()  # recomeça do zero a cada execução do notebook

engine = criar_engine()
sessao = nova_sessao(engine)

print(f"{VerdeClaro}Banco criado em:{Reset} {CinzaClaro}{CAMINHO_BANCO}{Reset}")
print(f"{VerdeClaro}Tamanho do arquivo:{Reset} {MagentaClaro}{CAMINHO_BANCO.stat().st_size} bytes{Reset}")


Banco criado em: /Users/lucaspaguettipereira/Documents/GitHub/Python_Impressionador_HashtagTreinamentos/Módulo 20 - Python + SQL/spec/hashtag_sqlalchemy.db
Tamanho do arquivo: 16384 bytes


## ➕ 4. Create — Inserindo com Objetos Python

Não existe `INSERT INTO` visível aqui: criar um objeto Python, adicionar na sessão (`session.add()`) e confirmar (`session.commit()`) já grava a linha no banco.

In [3]:
categoria_eletronicos = Categoria(nome="Eletrônicos")
categoria_papelaria = Categoria(nome="Papelaria")
sessao.add_all([categoria_eletronicos, categoria_papelaria])
sessao.commit()

produtos = [
    Produto(nome="Fone Bluetooth", preco=189.90, quantidade_estoque=25, categoria=categoria_eletronicos),
    Produto(nome="Carregador USB-C", preco=59.90, quantidade_estoque=40, categoria=categoria_eletronicos),
    Produto(nome="Caderno Universitário", preco=24.90, quantidade_estoque=100, categoria=categoria_papelaria),
]
sessao.add_all(produtos)
sessao.commit()

print(f"{VerdeClaro}2 categorias e {len(produtos)} produtos inseridos.{Reset}")


2 categorias e 3 produtos inseridos.


## 📖 5. Read — Consultando com a ORM

`sessao.query(Classe)` monta o `SELECT` — os filtros (`.filter()`), ordenações (`.order_by()`) e o `.all()`/`.first()` final funcionam parecido com o `pandas`, só que devolvendo objetos Python, não linhas cruas.

| Método 🔑 | Equivale a 🔓 |
|---|---|
| `sessao.query(Produto).all()` | `SELECT * FROM produtos` |
| `.filter(Produto.preco > 50)` | `WHERE preco > 50` |
| `.order_by(Produto.preco.desc())` | `ORDER BY preco DESC` |

In [4]:
todos_produtos = sessao.query(Produto).order_by(Produto.preco.desc()).all()

print(f"{CinzaClaro}Produtos cadastrados:{Reset}")
for produto in todos_produtos:
    print(f"  {VerdeClaro}{produto.nome}{Reset} — R${produto.preco} {CinzaEscuro}({produto.categoria.nome}){Reset}")

produtos_baratos = sessao.query(Produto).filter(Produto.preco < 100).all()
print(f"\n{CinzaClaro}Produtos abaixo de R$100:{Reset} {MagentaClaro}{[p.nome for p in produtos_baratos]}{Reset}")


Produtos cadastrados:
  Fone Bluetooth — R$189.90 (Eletrônicos)
  Carregador USB-C — R$59.90 (Eletrônicos)
  Caderno Universitário — R$24.90 (Papelaria)

Produtos abaixo de R$100: ['Carregador USB-C', 'Caderno Universitário']


## 🔄 6. Update e Delete

Update: buscar o objeto, mudar o atributo, `commit()`. Delete: buscar o objeto, `session.delete(objeto)`, `commit()`. Nenhum SQL explícito em nenhum dos dois.

In [5]:
fone = sessao.query(Produto).filter(Produto.nome == "Fone Bluetooth").first()
fone.preco = 159.90
sessao.commit()
print(f"{VerdeClaro}Fone Bluetooth atualizado{Reset} — novo preço: R${fone.preco}")

carregador = sessao.query(Produto).filter(Produto.nome == "Carregador USB-C").first()
sessao.delete(carregador)
sessao.commit()

restantes = sessao.query(Produto).all()
print(f"{VermelhoClaro}Carregador USB-C removido.{Reset} {CinzaClaro}Produtos restantes:{Reset} {MagentaClaro}{[p.nome for p in restantes]}{Reset}")

sessao.close()


Fone Bluetooth atualizado — novo preço: R$159.90
Carregador USB-C removido. Produtos restantes: ['Fone Bluetooth', 'Caderno Universitário']


## 🧾 Resumo do Módulo

| Notebook 🔑 | O que ficou pra trás 🔓 |
|---|---|
| 132–140 | Conceito de banco de dados + CRUD manual com `pyodbc` e SQL Server |
| 141–142 | Análise de dados real, conectando num banco maior |
| 143–148 | Sistema completo de controle de estoque (CRUD com regras de negócio) |
| 149 | O mesmo CRUD, agora em MySQL |
| 150 | Cheat sheet de SQL puro |
| 151 | SQLAlchemy: banco de dados criado inteiramente em Python, sem escrever um `CREATE TABLE` |

Do `CREATE TABLE` escrito à mão até classes Python que geram o banco sozinhas — o módulo fecha o ciclo de como o Python conversa com dado persistente, seja em SQL Server, MySQL ou SQLite.